# 01 · Thiết lập và phân tích khám phá (EDA)

Notebook này: (1) tải mã nguồn từ GitHub về Colab, (2) nối dữ liệu M5 từ Google Drive,
(3) chạy bộ kiểm thử chống rò rỉ, (4) phân tích khám phá phục vụ mục 6 của báo cáo:
tỷ lệ ngày bằng 0, phân nhóm ADI/CV², số ngày mỗi loại lịch trong từng cửa sổ, hiệu ứng tuần, sự kiện và SNAP.

Trước khi chạy: Runtime → Change runtime type → chọn **T4 GPU** (cần cho LSTM ở notebook 02).

In [ ]:
# ===== THIẾT LẬP CHUNG (chạy đầu tiên mỗi khi mở notebook) =====
GITHUB_USER = "<USERNAME>"          # <-- đổi thành tên tài khoản GitHub của bạn
REPO = "m5-calendar-forecasting"
DRIVE_DIR = "/content/drive/MyDrive/M5_project"   # thư mục trên Google Drive để lưu dữ liệu và kết quả

from google.colab import drive
drive.mount("/content/drive")

import os
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull
%cd /content/{REPO}
!pip install -q -e .
!mkdir -p {DRIVE_DIR}/data_raw {DRIVE_DIR}/results

In [ ]:
# ===== DỮ LIỆU M5 =====
# Cách A (khuyến nghị): bạn đã có 3 tệp CSV trong Google Drive tại {DRIVE_DIR}/data_raw
#   calendar.csv, sell_prices.csv, sales_train_evaluation.csv
# Cách B: tải từ Kaggle (bỏ dấu # ở các dòng dưới, cần tệp kaggle.json của bạn)
# from google.colab import files
# files.upload()                       # chọn kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !bash scripts/download_m5.sh {DRIVE_DIR}/data_raw

!mkdir -p data
!ln -sfn {DRIVE_DIR}/data_raw data/raw
!ls -lh data/raw/

## Kiểm thử chống rò rỉ
Mọi kiểm thử phải PASS trước khi chạy thí nghiệm. Hai kiểm thử mô hình chạy thật trên Colab vì có LightGBM và PyTorch.

In [ ]:
!pip install -q pytest
!python -m pytest -q

## Đọc dữ liệu và kiểm tra lược đồ

In [ ]:
import sys; sys.path.insert(0, "src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from m5cal.config import load_config
from m5cal.data import load_raw, build_panel
from m5cal.daytypes import calendar_features, daytype_matrix
from m5cal.sampling import demand_classes
from m5cal.splits import windows_from_config

cfg = load_config("configs")
raw = load_raw("data/raw")          # kiểm tra tên cột, giá trị âm, cột ngày liên tục
panel = build_panel(raw)
print("Số chuỗi:", panel.n, "| Số ngày doanh số:", panel.n_days, "| Số ngày lịch:", panel.P.shape[1])
print("Tỷ lệ ô bằng 0 (toàn bộ):", round(float((panel.Y == 0).mean()), 3))

## Phân nhóm nhu cầu (ADI/CV², chỉ tính trên dữ liệu đến d_1521)

In [ ]:
cls = demand_classes(panel, cfg["base"]["data"]["classify_until_d"])
meta = pd.concat([panel.meta, cls], axis=1)
display(meta.groupby(["cat_id", "demand_class"]).size().unstack(fill_value=0))
display(meta.groupby("store_id")["zero_share"].mean().round(3).to_frame("tỷ lệ ngày bằng 0"))

## Số ngày mỗi loại lịch trong từng cửa sổ (Bảng 9–10)
Lấy một chuỗi đại diện mỗi bang vì nhãn chỉ khác nhau giữa các bang do SNAP.

In [ ]:
calf = calendar_features(panel.calendar, cfg["base"]["daytypes"]["pre_event_days"])
windows = windows_from_config(cfg["base"]["windows"])
rep = meta.groupby("state_id").head(1).index.to_numpy()
sub = panel.subset(rep)
rows = []
for w in windows:
    lab = daytype_matrix(sub, calf, w.days)
    for k, st in enumerate(sub.meta["state_id"]):
        rows.append({"window": w.name, "role": w.role, "state": st, **pd.Series(lab[k]).value_counts().to_dict()})
counts = pd.DataFrame(rows).fillna(0).set_index(["window", "role", "state"])
display(counts.astype(int))

## Hiệu ứng tuần, sự kiện và SNAP (tổng doanh số mọi chuỗi)

In [ ]:
total = panel.Y.sum(axis=0)
cal = panel.calendar.iloc[: len(total)]
df = pd.DataFrame({"total": total, "weekday": cal["weekday"].values,
                   "event_type": cal["event_type_1"].fillna("none").values})
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
(df.groupby("weekday")["total"].mean().reindex(order) / df["total"].mean()).plot(kind="bar", ax=ax[0])
ax[0].set_title("Doanh số trung bình theo thứ (so với trung bình chung)"); ax[0].axhline(1, ls="--", c="gray")
(df.groupby("event_type")["total"].mean() / df.loc[df.event_type == "none", "total"].mean()).plot(kind="bar", ax=ax[1])
ax[1].set_title("Doanh số theo loại sự kiện (so với ngày không sự kiện)"); ax[1].axhline(1, ls="--", c="gray")
plt.tight_layout(); plt.show()

In [ ]:
# SNAP: so sánh doanh số FOODS ngày SNAP và không SNAP theo bang
res = []
for st in ["CA", "TX", "WI"]:
    rows_ = ((meta["state_id"] == st) & (meta["cat_id"] == "FOODS")).to_numpy()
    y = panel.Y[rows_].sum(axis=0)
    snap = cal[f"snap_{st}"].values == 1
    res.append({"state": st, "tỷ lệ SNAP/không SNAP": y[snap].mean() / y[~snap].mean()})
display(pd.DataFrame(res).round(3))